In [1]:
import sqlite3
import struct
from datetime import datetime

DB_PATH = "/data/cat.db"

def safe_float(v):
    if v is None:
        return 0.0
    if isinstance(v, bytes):
        try:
            return struct.unpack('<d', v)[0] if len(v) == 8 else 0.0
        except:
            return 0.0
    try:
        return float(v)
    except:
        return 0.0

def ts_to_human(ts):
    ts = safe_float(ts)
    unit = 1000.0 if ts > 1e11 else 1.0
    
    try:
        return datetime.fromtimestamp(ts / unit).strftime('%Y/%m/%d %H:%M:%S')
    except:
        return f"変換失敗({ts})"

conn = sqlite3.connect(DB_PATH, timeout=30)
conn.execute("PRAGMA journal_mode=WAL;")

# 全体の件数・範囲を確認
rows = conn.execute("SELECT COUNT(*), MIN(timestamp), MAX(timestamp) FROM raw_data").fetchone()
count, ts_min, ts_max = rows
print(f"総レコード数: {count}")
print(f"最古のデータ: {ts_to_human(ts_min)} (raw: {ts_min})")
print(f"最新のデータ: {ts_to_human(ts_max)} (raw: {ts_max})")
print()

# 直近50件を人間が読める形式で表示
print("=== 直近50件 ===")
recent = conn.execute(
    "SELECT timestamp, weight FROM raw_data ORDER BY timestamp DESC LIMIT 9000"
).fetchall()

for ts, w in recent:
    print(f"  {ts_to_human(ts)}  weight={safe_float(w):.1f}g")

print()

# 今日の日付のデータがあるか確認
today = datetime.now().strftime('%Y/%m/%d')
print(f"=== 今日({today})のデータ件数 ===")
today_rows = [r for r in recent if ts_to_human(r[0]).startswith(today)]
print(f"  直近50件のうち今日のデータ: {len(today_rows)}件")

if today_rows:
    print("  今日のデータあり ✅ → プログラム側の問題")
else:
    print("  今日のデータなし ❌ → センサー/収集側の問題の可能性")

conn.close()

総レコード数: 266068
最古のデータ: 2009/02/14 08:31:30 (raw: 1234567890)
最新のデータ: 2026/03/30 17:12:44 (raw: 1774858364)

=== 直近50件 ===
  2026/03/30 17:12:44  weight=63.8g
  2026/03/30 17:12:33  weight=63.2g
  2026/03/30 17:12:23  weight=61.6g
  2026/03/30 17:12:02  weight=63.0g
  2026/03/30 17:12:02  weight=63.0g
  2026/03/30 17:11:51  weight=63.9g
  2026/03/30 17:11:51  weight=63.9g
  2026/03/30 17:11:41  weight=63.8g
  2026/03/30 17:09:56  weight=64.7g
  2026/03/30 17:09:46  weight=64.8g
  2026/03/30 17:09:04  weight=64.5g
  2026/03/30 17:08:53  weight=64.9g
  2026/03/30 17:08:43  weight=65.6g
  2026/03/30 17:08:22  weight=65.5g
  2026/03/30 17:08:01  weight=65.3g
  2026/03/30 17:07:40  weight=65.3g
  2026/03/30 17:07:09  weight=65.0g
  2026/03/30 17:06:48  weight=66.3g
  2026/03/30 17:06:48  weight=66.3g
  2026/03/30 17:06:37  weight=64.4g
  2026/03/30 17:06:16  weight=65.5g
  2026/03/30 17:06:06  weight=65.7g
  2026/03/30 17:05:55  weight=64.1g
  2026/03/30 17:05:55  weight=64.1g
  2026/03/30 1